In [1]:
import os
import sys
# Adiciona o diretório pai (raiz do projeto) ao sys.path
sys.path.append(os.path.abspath(".."))

In [2]:
import numpy as np

In [3]:
import pandas as pd
import polars as pl
from preprocessing import pipeline as pipel

In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

c:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [5]:
dataf = pl.read_parquet(r'C:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification\data\dataset_enriquecido_10062026_v2.parquet')
df = dataf.to_pandas()

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8515 entries, 0 to 8514
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   TEMA_ORIGEM            8515 non-null   str    
 1   TEMA_CODIGO            8515 non-null   str    
 2   TOTAL_PROC_VINCULADOS  8277 non-null   float64
 3   ID_PROC                8515 non-null   int64  
 4   ques_direito           8277 non-null   str    
 5   tese_firmada           5768 non-null   str    
 6   info_legislativa       3307 non-null   str    
 7   ementa                 8019 non-null   str    
 8   texto                  8140 non-null   str    
 9   inteiro_teor           8515 non-null   str    
dtypes: float64(1), int64(1), str(8)
memory usage: 2.5 GB


In [7]:
df['TEMA'] = df['TEMA_ORIGEM'] + '_' + df['TEMA_CODIGO'].astype(str)
df['TEMA'].value_counts()

TEMA
STF_1177    100
STF_1218    100
STF_1266    100
STF_1417    100
STF_163     100
           ... 
STJ_1101     10
STJ_1115     10
STJ_1266     10
STJ_1290     10
STJ_766      10
Name: count, Length: 159, dtype: int64

In [8]:
df['ementa'] = df['ementa'].fillna("")
df['info_legislativa'] = df['info_legislativa'].fillna("")

In [9]:
df['ementa_limpa'] = df['ementa'].apply(lambda x: pipel.aplicar_pipeline_texto(x))
df['info_legislativa_limpa'] = df['info_legislativa'].apply(lambda x: pipel.aplicar_pipeline_texto(x))
df['inteiro_teor_limpo'] = df['inteiro_teor'].apply(lambda x: pipel.aplicar_pipeline_texto(x))

In [10]:
model_name = "dominguesm/legal-bert-base-cased-ptbr"
num_classes = 3  # Adjust based on your task

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_classes)

ImportError: 
AutoModelForSequenceClassification requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.


In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# Assuming you have a Hugging Face Dataset 'dataset'
tokenized_datasets = df.map(tokenize_function, batched=True)